# 07 • PDF RAG: คู่มืองานคลังและการส่งมอบ

เป้าหมาย: อัปโหลด PDF ภาษาไทย แยกข้อความ เก็บเลขหน้า สร้าง Embedding และตอบพร้อมอ้างอิงหน้า

**ข้อมูลสมมติสำหรับการอบรม:** ชื่อหน่วยงาน Lot ผู้ขาย ค่าที่วัด และ SOP ในตัวอย่างถูกสร้างขึ้นเพื่อเรียนรู้ AI ไม่ใช่ข้อมูลการผลิตหรือระเบียบจริงของกลุ่มศรีตรัง

บริบทงานอ้างอิงจาก [เว็บไซต์กลุ่มศรีตรัง](https://www.sritranggroup.com/th/home) ส่วนรูปแบบ notebook อ้างอิงจาก [ตัวอย่าง GitHub](https://github.com/biodatlab/cpf-genai-workshop/tree/main/tutorial_notebooks) และใช้ [OpenAI API documentation](https://developers.openai.com/api/docs) สำหรับ syntax

## เตรียมพร้อมก่อนเริ่ม
เพิ่ม Secret ชื่อ `OPENAI_API_KEY` ใน Colab ด้วยตนเอง และเปิด **Notebook access** สำหรับ notebook นี้ โค้ดจะอ่านผ่าน `userdata.get()` โดยไม่แสดงค่า key

รันเซลล์จากบนลงล่าง หลังติดตั้ง package และเตรียม key แล้ว ทดสอบตัวอย่างหลักใน Colab ด้วย API แล้ว แต่ผล LLM อาจเปลี่ยนได้ การผ่านตัวอย่างนี้ไม่ใช่การรับรองทุกคำถามหรือการใช้งานจริง

ใช้ `gpt-4.1-mini` สำหรับสร้างคำตอบ (ไม่เกิน 500 tokens ต่อ request) และ `text-embedding-3-small` สำหรับ Embedding โดย Agent Loop จำกัดไว้ 5 รอบ

In [1]:
%pip -q install openai numpy pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 45.0 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
MODEL = "gpt-4.1-mini"

## อัปโหลดเอกสารและอ่านข้อความ
ใช้ sritrang_training_sop_th.pdf ตัวอย่างสมมติ อัปโหลดผ่าน Files pane ทางซ้ายก่อนรันได้ หรือใช้ปุ่มเลือกไฟล์ใน output หากไม่มีไฟล์ใน runtime ต้องใช้ PDF ที่มี text layer

In [4]:
from google.colab import files
import pymupdf

from pathlib import Path
# อัปโหลดผ่าน Files pane ได้ก่อนรัน หรือใช้ปุ่มเลือกไฟล์ด้านล่าง
sample = Path("sritrang_training_sop_th.pdf")
uploaded = {sample.name: sample.read_bytes()} if sample.exists() else files.upload()
pdf_name = next(iter(uploaded))
doc = pymupdf.open(stream=uploaded[pdf_name], filetype="pdf")
chunks = []
for page_number, page in enumerate(doc, 1):
    text = page.get_text()
    for start in range(0, len(text), 600):
        chunks.append(f"[หน้า {page_number}] " + text[start:start + 750])
if not chunks:
    raise ValueError("ไม่พบข้อความ กรุณาใช้ PDF ที่มี text layer")
print("จำนวนหน้า:", len(doc), "จำนวน chunks:", len(chunks))
print(chunks[0])

จำนวนหน้า: 3 จำนวน chunks: 5
[หน้า 1] เอกสารสมมติสำหรับอบรม AI
หน้า 1
คู่มือฝึกอบรม: รับวัตถุดิบและคลังสินค้า
ตัวอย่างสำหรับพนักงานศรีตรัง | ข้อมูลสมมติสำหรับฝึกใช้ AI เท่านั้น ไม่ใช่ SOP จริงของบริษัท
[SOP-1 รับน้ำยาง] บันทึก Lot ผู้ขาย น้ำหนักสุทธิ ทะเบียนรถ และเวลารับเข้า
เก็บรหัสผู้ขายเพื่อเชื่อมโยงข้อมูล Traceability
[SOP-2 QC] เก็บผล DRC พร้อมรหัสตัวอย่างและ Lot หากผลยังไม่ครบ ให้ระบุสถานะรอ QC
ห้ามสรุปว่าผ่านเกณฑ์จากข้อมูลที่หายไป
[SOP-3 คลังสินค้า] Lot ที่รอ QC ต้องแยกในพื้นที่พักรอผลและติดป้าย Lot
ตรวจสถานะก่อนยืนยันการจัดส่ง
ตัวอย่างใบรับน้ำยาง: Lot LTX001 ผู้ขาย สวนยางตัวอย่าง น้ำหนักสุทธิ 8,000 กก. DRC 32.5%
สถานะรอ QC



In [5]:
import numpy as np

EMBED_MODEL = "text-embedding-3-small"

def embed(texts):
    result = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([item.embedding for item in result.data])

vectors = embed(chunks)
print("ขนาด Embedding matrix:", vectors.shape)

ขนาด Embedding matrix: (5, 1536)


In [6]:
def retrieve(question, k=2):
    query = embed([question])[0]
    scores = vectors @ query / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(query))
    indices = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in indices]

In [8]:
question = "เมื่อ Lot ยังไม่มีผล QC ฝ่ายคลังและฝ่ายขนส่งต้องทำอะไรตามคู่มือนี้?"
hits = retrieve(question, k=3)
context = "\n".join(text for text, score in hits)
print(context)
print(client.responses.create(
    model=MODEL,
    instructions="ตอบเป็นภาษาไทยโดยใช้เฉพาะข้อความที่ให้ อ้างเลขหน้า หากข้อมูลไม่มีให้ระบุว่าไม่พบในเอกสาร ห้ามอนุมานขั้นตอนเพิ่ม หากเอกสารไม่ได้ระบุหน้าที่ฝ่ายขนส่งโดยตรง ให้บอกว่าไม่ได้ระบุหน้าที่ฝ่ายขนส่งโดยตรง",
    input=f"ข้อความที่ค้นพบ:\n{context}\n\nคำถาม: {question}",
    max_output_tokens=500,
).output_text)

[หน้า 2] เอกสารสมมติสำหรับอบรม AI
หน้า 2
ตาราง Lot และผังพื้นที่คลัง
พื้นที่และสถานะต่อไปนี้เป็นตัวอย่างสมมติ
Lot
ผลิตภัณฑ์
สถานะ
พื้นที่
LTX001
น้ำยาง
รอ QC
พื้นที่พักรอผล
STR20001
ยางแท่ง STR 20
ผ่าน QC
พื้นที่พร้อมส่ง
ผังตัวอย่าง (อ่านจากซ้ายไปขวา)
รับเข้า
→พักรอ QC
→ผ่าน QC
→จุดโหลด
Lot LTX001 ต้องอยู่พื้นที่พักรอผล QC และติดป้าย Lot ก่อนยืนยันการจัดส่ง

[หน้า 1] เอกสารสมมติสำหรับอบรม AI
หน้า 1
คู่มือฝึกอบรม: รับวัตถุดิบและคลังสินค้า
ตัวอย่างสำหรับพนักงานศรีตรัง | ข้อมูลสมมติสำหรับฝึกใช้ AI เท่านั้น ไม่ใช่ SOP จริงของบริษัท
[SOP-1 รับน้ำยาง] บันทึก Lot ผู้ขาย น้ำหนักสุทธิ ทะเบียนรถ และเวลารับเข้า
เก็บรหัสผู้ขายเพื่อเชื่อมโยงข้อมูล Traceability
[SOP-2 QC] เก็บผล DRC พร้อมรหัสตัวอย่างและ Lot หากผลยังไม่ครบ ให้ระบุสถานะรอ QC
ห้ามสรุปว่าผ่านเกณฑ์จากข้อมูลที่หายไป
[SOP-3 คลังสินค้า] Lot ที่รอ QC ต้องแยกในพื้นที่พักรอผลและติดป้าย Lot
ตรวจสถานะก่อนยืนยันการจัดส่ง
ตัวอย่างใบรับน้ำยาง: Lot LTX001 ผู้ขาย สวนยางตัวอย่าง น้ำหนักสุทธิ 8,000 กก. DRC 32.5%
สถานะรอ QC

[หน้า 3]  สิ่งแวดล้อม] หากพบ

## ลองทำด้วยตนเอง
1. ถามขั้นตอนจัดการ Lot ที่ยังไม่มีผล QC
2. ถามเรื่องการเลื่อนรถรับสินค้าระหว่างน้ำท่วม แล้วตรวจเลขหน้าใน PDF
3. ถามเรื่องที่ไม่มีในคู่มือ ตรวจ chunks ที่ได้ก่อนประเมินคำตอบ